In [1]:
import numpy as np
import time
import json

from numba import njit, prange

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import shuffle

import torch
import torch.nn as nn
import torch.optim as optim


# =========================================================
# CONSTANTS & ARITHMETIC (Numba Optimized)
# =========================================================

KYBER_Q = 3329
QINV = -3327

HW_LOOKUP = np.array(
    [bin(i).count("1") for i in range(256)],
    dtype=np.uint8
)

@njit(inline='always')
def int16_numba(x):
    x = x & 0xFFFF
    return x - 0x10000 if x & 0x8000 else x

@njit(inline='always')
def uint16_numba(x):
    return x & 0xFFFF

@njit(inline='always')
def montgomery_reduce_numba(a):
    t = int16_numba(a)
    t = int16_numba(t * QINV)
    res = (a - (int(t) * KYBER_Q)) >> 16
    return int16_numba(res)

@njit(inline='always')
def fqmul_numba(a, b):
    prod = int16_numba(a) * int16_numba(b)
    return uint16_numba(montgomery_reduce_numba(prod))

@njit(parallel=True)
def compute_prod_chunk(nonce_b, g_chunk, att, cols):
    prod_chunk = np.empty((att, cols), dtype=np.uint16)
    for j in prange(att):
        b = int(nonce_b[j])
        for k in range(cols):
            prod_chunk[j, k] = fqmul_numba(int(g_chunk[k]), b)
    return prod_chunk

@njit(parallel=True)
def compute_tot_loglike(class_chunk, logprobs, att, cols):
    out = np.zeros(cols, dtype=np.float64)
    for k in prange(cols):
        s = 0.0
        for j in range(att):
            s += logprobs[j, class_chunk[j, k]]
        out[k] = s
    return out

# =========================================================
# LOADERS & COMPRESSION
# =========================================================

def load_traces(path):
    return np.load(path).astype(np.float32)

def load_nonces(path):
    nonces = np.load(path)
    if nonces.dtype == np.uint8:
        if nonces.shape[1] % 2 != 0:
            raise ValueError("Nonce byte array must have even number of columns")
        nonces16 = nonces[:, ::2].astype(np.uint16) + (nonces[:, 1::2].astype(np.uint16) << 8)
        return nonces16
    return nonces.astype(np.uint16)

def compress(raw, cf):
    N, L = raw.shape
    CL = (L + cf - 1) // cf
    out = np.zeros((N, CL), dtype=np.float32)
    for i in range(CL):
        start = i * cf
        end = min(L, (i + 1) * cf)
        out[:, i] = raw[:, start:end].mean(axis=1)
    return out


# =========================================================
# PYTORCH LSTM ARCHITECTURE
# =========================================================

class RNNNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(RNNNet, self).__init__()
        
        # RNN Layer
        # batch_first=True expects input shape: (Batch, Timesteps, Features)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, batch_first=True)
        
        # Dense Classification Head
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # out shape: (Batch, Timesteps, Hidden_Dim)
        # we only care about the hidden state output of the very last timestep: out[:, -1, :]
        out, hn = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out


# =========================================================
# MAIN ATTACK VIA PYTORCH LSTM (GPU-ACCELERATED)
# =========================================================

def attack_py_rnn_hw_multiclass(
    traces,
    nonces,
    prof,
    which,
    window_start,      
    window_end,
    chunk=1024,
    random_state=42,
):
    N = traces.shape[0]
    att = N - prof

    if att <= 0:
        raise ValueError("Number of attack traces is zero or negative.")

    idx = 2 if which == 0 else 5
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # =====================================================
    # LABELS
    # =====================================================
    labels = np.zeros(prof, dtype=np.int8)
    for i in range(prof):
        val = int(nonces[i, idx])
        msb = (val >> 8) & 0xFF
        labels[i] = int(HW_LOOKUP[msb])

    le = LabelEncoder()
    labels_encoded = le.fit_transform(labels)
    num_classes = len(le.classes_)

    # =====================================================
    # SEQUENTIAL DATA PREPARATION
    # =====================================================
    X_train_raw = traces[:prof, window_start:window_end]
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_raw)

    X_train_t = torch.tensor(X_train_s, dtype=torch.float32).unsqueeze(2).to(device)
    y_train_t = torch.tensor(labels_encoded, dtype=torch.long).to(device)

    torch.manual_seed(random_state)

    model = RNNNet(input_dim=1, hidden_dim=32, output_dim=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

    model.train()
    for epoch in range(10):  
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # =====================================================
    # FIXED ATTACK PHASE (Batching Inference to Save VRAM)
    # =====================================================
    X_attack_raw = traces[prof:, window_start:window_end]
    X_attack_s = scaler.transform(X_attack_raw)
    
    # Keep attack tensor on CPU initially
    X_attack_t = torch.tensor(X_attack_s, dtype=torch.float32).unsqueeze(2)

    # Empty list to collect chunked probability predictions
    all_probs = []
    inference_batch_size = 64  # Process 64 traces at a time on GPU instead of thousands

    model.eval()
    with torch.no_grad():
        for i in range(0, att, inference_batch_size):
            # Take a small slice of attack data and push it to GPU
            batch_x = X_attack_t[i : i + inference_batch_size].to(device)
            
            logits = model(batch_x)
            probs_t = torch.softmax(logits, dim=1)
            
            # Immediately pull back to CPU numpy array to keep VRAM perfectly clean
            all_probs.append(probs_t.cpu().numpy())

    # Combine batched pieces back into one full master probability matrix
    probs = np.vstack(all_probs)

    probs = np.clip(probs, 1e-15, 1.0)
    logprobs = np.log(probs)

    nonce_b = nonces[prof:, 1].astype(np.int64)
    guesses = np.arange(65536, dtype=np.int64)
    all_scores = np.full(65536, -np.inf, dtype=np.float64)

    # =====================================================
    # BUILD HW -> CLASS MAP ONCE
    # =====================================================
    classes_present = np.asarray(le.classes_, dtype=np.int16)
    hw_to_col = np.empty(256, dtype=np.int64)

    for hw_val in range(256):
        diffs = np.abs(classes_present - hw_val)
        hw_to_col[hw_val] = np.argmin(diffs)

    # =====================================================
    # CHUNK PROCESSING
    # =====================================================
    for start in range(0, 65536, chunk):
        end = min(start + chunk, 65536)
        g_chunk = guesses[start:end]
        cols = g_chunk.shape[0]

        prod_chunk = compute_prod_chunk(nonce_b, g_chunk, att, cols)
        msb_chunk = ((prod_chunk >> 8) & 0xFF).astype(np.uint8)
        hw_chunk = HW_LOOKUP[msb_chunk]
        class_chunk = hw_to_col[hw_chunk]

        tot_loglike = compute_tot_loglike(class_chunk, logprobs, att, cols)
        all_scores[start:end] = tot_loglike

    return all_scores

# =========================================================
# EVALUATION METRIC
# =========================================================

def tie_inclusive_rank(all_scores, true_guess_index):
    true_score = all_scores[int(true_guess_index)]
    return int(np.sum(all_scores >= true_score))


# =========================================================
# MAIN LOOP
# =========================================================

if __name__ == "__main__":

    traces_file = "traces.npy"
    nonces_file = "nonces.npy"

    compressF = 10
    profFrac = 0.8
    CHUNK = 1024
    RANDOM_STATE = 42

    print("\n--- Optimized PyTorch LSTM Continuous Window Attack ---\n")

    raw = load_traces(traces_file)
    nonces = load_nonces(nonces_file)

    raw, nonces = shuffle(raw, nonces, random_state=RANDOM_STATE)

    N_total = min(raw.shape[0], nonces.shape[0])
    raw = raw[:N_total]
    nonces = nonces[:N_total]

    results = []

    for N in range(300, 2301, 500):

        if N > len(raw):
            print(f"{N} SKIPPED")
            continue

        print(f"Running N={N}")

        raw_subset = raw[:N]
        nonces_subset = nonces[:N]

        traces = compress(raw_subset, compressF)
        prof = max(1, int(profFrac * N))
        att = N - prof

        t_start = time.process_time()

        # Execute asymmetric convolutional profiles
        scores_a0 = attack_py_rnn_hw_multiclass(
            traces,
            nonces_subset,
            prof,
            which=0,
            window_start=1000,    # Target range for a0: 1000 to 2000
            window_end=2000,
            chunk=CHUNK,
            random_state=RANDOM_STATE,
        )

        scores_a1 = attack_py_rnn_hw_multiclass(
            traces,
            nonces_subset,
            prof,
            which=1,
            window_start=3000,    # Target range for a1: 3000 to 4500
            window_end=4500,
            chunk=CHUNK,
            random_state=RANDOM_STATE,
        )

        t_end = time.process_time()
        cpu_time = t_end - t_start

        true0 = int(nonces_subset[0, 0])
        true1 = int(nonces_subset[0, 3])

        rank_a0 = tie_inclusive_rank(scores_a0, true0)
        rank_a1 = tie_inclusive_rank(scores_a1, true1)

        result = {
            "N": int(N),
            "prof": int(prof),
            "att": int(att),
            "rank_a0": int(rank_a0),
            "rank_a1": int(rank_a1),
            "cpu_time": float(cpu_time),
        }

        results.append(result)
        print(result)



--- Optimized PyTorch LSTM Continuous Window Attack ---

Running N=300
{'N': 300, 'prof': 240, 'att': 60, 'rank_a0': 30853, 'rank_a1': 3810, 'cpu_time': 7.9375}
Running N=800
{'N': 800, 'prof': 640, 'att': 160, 'rank_a0': 1331, 'rank_a1': 5382, 'cpu_time': 4.703125}
Running N=1300
{'N': 1300, 'prof': 1040, 'att': 260, 'rank_a0': 25336, 'rank_a1': 12139, 'cpu_time': 7.171875}
Running N=1800
{'N': 1800, 'prof': 1440, 'att': 360, 'rank_a0': 17339, 'rank_a1': 39880, 'cpu_time': 9.75}
Running N=2300
{'N': 2300, 'prof': 1840, 'att': 460, 'rank_a0': 34064, 'rank_a1': 61759, 'cpu_time': 12.859375}
